# Semantic Cluster Prep Orchestrator

This notebook builds per-patient fastText embeddings and semantic category predictions from the row-aligned `*_transcripts_with_neural.csv` tables.

Outputs per patient:
- `embeddings/fasttext_word_embeddings.npy`
- `embeddings/semantic_cluster_predictions.npy`
- `embeddings/semantic_cluster_manifest.json`
- `decoding_inputs/all_words_filtered.csv`

The worker reads the raw word column only long enough to compute the numeric artifacts, and the saved convenience CSV intentionally excludes text-bearing columns.


In [1]:
import subprocess
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [2]:
# -- Config -----------------------------------------------------------------
PATIENTS = None  # None -> auto-discover from vad_new/*/*_transcripts_with_neural.csv

VAD_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

PYTHON_BIN = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/semantic_cluster_worker.py')

FASTTEXT_MODEL_OVERRIDE = None
CLASSIFIER_OVERRIDE = None
BATCH_SIZE = 50000
FORCE = False

print('worker:', WORKER_PATH)


worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/semantic_cluster_worker.py


In [3]:
# -- Helpers ----------------------------------------------------------------
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def discover_patients() -> list[str]:
    patients = []
    for patient_dir in sorted(VAD_ROOT.iterdir()):
        if not patient_dir.is_dir():
            continue
        patient = patient_dir.name
        if (patient_dir / f'{patient}_transcripts_with_neural.csv').exists():
            patients.append(patient)
    return patients


def patient_root(patient: str) -> Path:
    return VAD_ROOT / patient


def resolve_patient(patient: str) -> dict:
    root = patient_root(patient)
    embeddings_dir = root / 'embeddings'
    decoding_inputs_dir = root / 'decoding_inputs'
    transcript_csv = first_existing([
        root / f'{patient}_transcripts_with_neural.csv',
        root / f'{patient}_transcripts.csv',
    ])
    word_frs_path = first_existing([root / 'neural_embeddings' / 'word_frs.npy'])
    fasttext_model = first_existing([
        FASTTEXT_MODEL_OVERRIDE,
        VAD_ROOT / 'cc.en.300.bin',
        LEGACY_ROOT / 'cc.en.300.bin',
    ])
    classifier_pkl = first_existing([
        CLASSIFIER_OVERRIDE,
        VAD_ROOT / 'semantic_cat_classifier.pkl',
        LEGACY_ROOT / 'semantic_cat_classifier.pkl',
    ])
    return {
        'patient': patient,
        'patient_root': root,
        'transcript_csv': transcript_csv,
        'word_frs_path': word_frs_path,
        'fasttext_model': fasttext_model,
        'classifier_pkl': classifier_pkl,
        'embeddings_dir': embeddings_dir,
        'decoding_inputs_dir': decoding_inputs_dir,
        'embedding_path': embeddings_dir / 'fasttext_word_embeddings.npy',
        'cluster_preds_path': embeddings_dir / 'semantic_cluster_predictions.npy',
        'manifest_path': embeddings_dir / 'semantic_cluster_manifest.json',
        'success_path': embeddings_dir / '_SUCCESS',
        'error_path': embeddings_dir / 'semantic_cluster_error.txt',
        'stdout_log': embeddings_dir / 'semantic_cluster_stdout.log',
        'stderr_log': embeddings_dir / 'semantic_cluster_stderr.log',
        'safe_df_path': decoding_inputs_dir / 'all_words_filtered.csv',
    }


PATIENTS = discover_patients() if PATIENTS is None else PATIENTS
print('patients:', PATIENTS)


def build_status_row(patient: str) -> dict:
    info = resolve_patient(patient)
    return {
        'patient': patient,
        'transcript_csv': info['transcript_csv'],
        'word_frs_path': info['word_frs_path'],
        'fasttext_model': info['fasttext_model'],
        'classifier_pkl': info['classifier_pkl'],
        'embedding_path': info['embedding_path'],
        'cluster_preds_path': info['cluster_preds_path'],
        'safe_df_path': info['safe_df_path'],
        'has_transcript_csv': info['transcript_csv'] is not None and info['transcript_csv'].exists(),
        'has_word_frs': info['word_frs_path'] is not None and info['word_frs_path'].exists(),
        'has_fasttext_model': info['fasttext_model'] is not None and info['fasttext_model'].exists(),
        'has_classifier_pkl': info['classifier_pkl'] is not None and info['classifier_pkl'].exists(),
        'has_embedding_npy': info['embedding_path'].exists(),
        'has_cluster_preds_npy': info['cluster_preds_path'].exists(),
        'has_manifest': info['manifest_path'].exists(),
        'has_safe_df': info['safe_df_path'].exists(),
        'has_success': info['success_path'].exists(),
        'has_error': info['error_path'].exists(),
        'ready': all([
            info['transcript_csv'] is not None and info['transcript_csv'].exists(),
            info['word_frs_path'] is not None and info['word_frs_path'].exists(),
            info['fasttext_model'] is not None and info['fasttext_model'].exists(),
            info['classifier_pkl'] is not None and info['classifier_pkl'].exists(),
        ]),
    }


status_df = pd.DataFrame([build_status_row(patient) for patient in PATIENTS])
status_df


patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFS', 'YFT', 'YFU', 'YFV']


,patient,transcript_csv,word_frs_path,fasttext_model,classifier_pkl,embedding_path,cluster_preds_path,safe_df_path,has_transcript_csv,has_word_frs,has_fasttext_model,has_classifier_pkl,has_embedding_npy,has_cluster_preds_npy,has_manifest,has_safe_df,has_success,has_error,ready
0,YEY,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,False,False,False,False,False,False,True
1,YEZ,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,False,False,False,False,False,False,True
2,YFA,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,False,False,False,False,False,False,True
3,YFB,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
4,YFC,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
5,YFD,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
6,YFE,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
7,YFF,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
8,YFG,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True

In [4]:
# -- Run per-patient semantic prep ------------------------------------------
RUN_PATIENTS = PATIENTS

ran = []
failed = []

for patient in RUN_PATIENTS:
    info = resolve_patient(patient)
    row = build_status_row(patient)
    if not row['ready']:
        print(f'skipping {patient}: missing required inputs')
        continue
    if row['has_success'] and not FORCE:
        print(f'already done: {patient}')
        continue

    cmd = [
        PYTHON_BIN,
        str(WORKER_PATH),
        '--patient', patient,
        '--transcripts-csv', str(info['transcript_csv']),
        '--fasttext-model', str(info['fasttext_model']),
        '--classifier-pkl', str(info['classifier_pkl']),
        '--embeddings-dir', str(info['embeddings_dir']),
        '--decoding-inputs-dir', str(info['decoding_inputs_dir']),
        '--batch-size', str(BATCH_SIZE),
    ]
    if FORCE:
        cmd.append('--force')

    info['embeddings_dir'].mkdir(parents=True, exist_ok=True)
    print(f'running semantic prep for {patient}')
    res = subprocess.run(cmd, capture_output=True, text=True)
    info['stdout_log'].write_text(res.stdout)
    info['stderr_log'].write_text(res.stderr)

    if res.returncode == 0:
        ran.append(patient)
        print(f'done: {patient}')
    else:
        failed.append(patient)
        print(f'FAILED: {patient}')
        print(res.stderr[-2000:])

print('ran:', ran)
print('failed:', failed)


running semantic prep for YEY
done: YEY
running semantic prep for YEZ
done: YEZ
running semantic prep for YFA
done: YFA
already done: YFB
already done: YFC
already done: YFD
already done: YFE
already done: YFF
already done: YFG
already done: YFI
already done: YFK
already done: YFL
already done: YFM
already done: YFN
already done: YFP
already done: YFQ
already done: YFR
already done: YFS
already done: YFT
already done: YFU
already done: YFV
ran: ['YEY', 'YEZ', 'YFA']
failed: []


In [5]:
# -- Refresh status ----------------------------------------------------------
status_df = pd.DataFrame([build_status_row(patient) for patient in PATIENTS])
status_df


,patient,transcript_csv,word_frs_path,fasttext_model,classifier_pkl,embedding_path,cluster_preds_path,safe_df_path,has_transcript_csv,has_word_frs,has_fasttext_model,has_classifier_pkl,has_embedding_npy,has_cluster_preds_npy,has_manifest,has_safe_df,has_success,has_error,ready
0,YEY,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
1,YEZ,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
2,YFA,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
3,YFB,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
4,YFC,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
5,YFD,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
6,YFE,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
7,YFF,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
8,YFG,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,True,True,True,True,True,True,False,True
9,YFI,/mnt/lab

In [6]:
# -- Alignment check ---------------------------------------------------------
import numpy as np

for patient in PATIENTS:
    info = resolve_patient(patient)
    if not (info['cluster_preds_path'].exists() and info['embedding_path'].exists() and info['word_frs_path'] is not None and info['word_frs_path'].exists()):
        continue
    preds = np.load(info['cluster_preds_path'], mmap_mode='r')
    embeds = np.load(info['embedding_path'], mmap_mode='r')
    frs = np.load(info['word_frs_path'], mmap_mode='r')
    print(patient, {
        'cluster_preds_shape': preds.shape,
        'embedding_shape': embeds.shape,
        'frs_shape': frs.shape,
        'safe_df_exists': info['safe_df_path'].exists(),
    })


YEY {'cluster_preds_shape': (47540,), 'embedding_shape': (47540, 300), 'frs_shape': (47540, 16), 'safe_df_exists': True}
YEZ {'cluster_preds_shape': (89171,), 'embedding_shape': (89171, 300), 'frs_shape': (89171, 24), 'safe_df_exists': True}
YFA {'cluster_preds_shape': (610928,), 'embedding_shape': (610928, 300), 'frs_shape': (610928, 40), 'safe_df_exists': True}
YFB {'cluster_preds_shape': (159410,), 'embedding_shape': (159410, 300), 'frs_shape': (159410, 40), 'safe_df_exists': True}
YFC {'cluster_preds_shape': (335602,), 'embedding_shape': (335602, 300), 'frs_shape': (335602, 64), 'safe_df_exists': True}
YFD {'cluster_preds_shape': (442808,), 'embedding_shape': (442808, 300), 'frs_shape': (442808, 32), 'safe_df_exists': True}
YFE {'cluster_preds_shape': (227582,), 'embedding_shape': (227582, 300), 'frs_shape': (227582, 16), 'safe_df_exists': True}
YFF {'cluster_preds_shape': (264186,), 'embedding_shape': (264186, 300), 'frs_shape': (264186, 48), 'safe_df_exists': True}
YFG {'cluster_